# Lab 5.5 &mdash; Tool Routing: Dispatch an Action to a Tool

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Look a tool up by name in the registry
- Run it and capture the observation the agent will read next
- Handle an unknown tool (and a failing tool) without crashing

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

**Reference:** [Module 5 slides &mdash; Anatomy of an agent](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-05-05"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
Once the agent has chosen an **action name**, the orchestrator must **route** it to the matching
tool, run it, and wrap the result as an **observation**. Real agents hallucinate tool names, so
routing must fail **safely**: an unknown tool returns a message, not a crash.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)
KNOWLEDGE = {"capital of france": "Paris", "population of metropolis": "120000"}
REGISTRY = {
    "calculator": {"name": "calculator", "fn": safe_calc},
    "lookup":     {"name": "lookup", "fn": lambda k: KNOWLEDGE.get(k.lower().strip(), "unknown")},
    "weather":    {"name": "weather", "fn": lambda city: "sunny 24C in " + city.strip()},
}
print("registry:", list(REGISTRY))
# Note: a real tool can also FAIL -- e.g. calculator on non-math input raises. Routing must survive that.

## Your Turn
Implement `route`: find the tool, run it, return a dict. Route must survive **three** hazards a real
agent hits &mdash; an **unknown** tool name, a tool that **raises** (bad input), and still pass a good
call through. The `try/except` is what turns a crash into a recoverable observation.

In [ ]:
def route(registry, action, arg):
    tool = ___    # TODO: get the tool by name (None if it is not registered)
    if tool is None:
        return {"ok": False, "observation": ___}   # TODO: a message naming the unknown tool
    try:
        result = ___   # TODO: run the tool function on arg
        return {"ok": True, "observation": result}
    except Exception as e:
        return {"ok": False, "observation": "tool error: " + type(e).__name__}

try:
    print(route(REGISTRY, 'calculator', '10/2'))
    print(route(REGISTRY, 'weather', 'tokyo'))
    print(route(REGISTRY, 'no_such_tool', 'x'))    # unknown tool
    print(route(REGISTRY, 'calculator', 'not math'))  # a tool that RAISES
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("routes the calculator correctly", lambda: route(REGISTRY, "calculator", "10/2")["observation"] == 5.0)
expect_true("routes the lookup correctly", lambda: route(REGISTRY, "lookup", "capital of france")["observation"] == "Paris")
expect_true("routes the third tool (weather) correctly", lambda: "tokyo" in route(REGISTRY, "weather", "tokyo")["observation"])
expect_true("ok flag is True on success", lambda: route(REGISTRY, "calculator", "2+2")["ok"] is True)
expect_true("unknown tool handled without crashing", lambda: route(REGISTRY, "no_such_tool", "x")["ok"] is False)
expect_true("unknown tool message names the tool", lambda: "no_such_tool" in route(REGISTRY, "no_such_tool", "x")["observation"])
expect_true("a tool that RAISES is caught, not crashed", lambda: route(REGISTRY, "calculator", "not math")["ok"] is False)
expect_true("the caught error is reported as an observation", lambda: "error" in route(REGISTRY, "calculator", "not math")["observation"].lower())

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Routing turns a chosen action into a real observation &mdash; safely. The split between deciding (the policy) and executing (the router) is what makes agents debuggable.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>